# Tantangan: Menganalisis Teks tentang Ilmu Data

Dalam contoh ini, mari lakukan latihan sederhana yang mencakup semua langkah dari proses ilmu data tradisional. Anda tidak perlu menulis kode apa pun, Anda cukup mengklik sel di bawah ini untuk menjalankannya dan mengamati hasilnya. Sebagai tantangan, Anda didorong untuk mencoba kode ini dengan data yang berbeda.

## Tujuan

Dalam pelajaran ini, kita telah membahas berbagai konsep terkait Ilmu Data. Mari coba temukan lebih banyak konsep terkait dengan melakukan **penambangan teks**. Kita akan mulai dengan teks tentang Ilmu Data, mengekstrak kata kunci darinya, kemudian mencoba memvisualisasikan hasilnya.

Sebagai teks, saya akan menggunakan halaman tentang Ilmu Data dari Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Langkah 1: Mendapatkan Data

Langkah pertama dalam setiap proses data science adalah mendapatkan data. Kita akan menggunakan pustaka `requests` untuk melakukan itu:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Langkah 2: Mengubah Data

Langkah berikutnya adalah mengubah data ke dalam bentuk yang sesuai untuk diproses. Dalam kasus kami, kami telah mengunduh kode sumber HTML dari halaman tersebut, dan kami perlu mengubahnya menjadi teks biasa.

Ada banyak cara untuk melakukan ini. Kami akan menggunakan [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), sebuah pustaka Python populer untuk mengurai HTML. BeautifulSoup memungkinkan kami menargetkan elemen HTML tertentu, sehingga kami dapat fokus pada konten artikel utama dari Wikipedia dan mengurangi beberapa menu navigasi, sidebar, footer, dan konten lain yang tidak relevan (meskipun beberapa teks boilerplate mungkin masih tersisa).


Pertama, kita perlu menginstal perpustakaan BeautifulSoup untuk parsing HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Langkah 3: Mendapatkan Wawasan

Langkah terpenting adalah mengubah data kita menjadi suatu bentuk dari mana kita dapat menarik wawasan. Dalam kasus kami, kami ingin mengekstrak kata kunci dari teks, dan melihat kata kunci mana yang lebih bermakna.

Kami akan menggunakan pustaka Python yang disebut [RAKE](https://github.com/aneesha/RAKE) untuk ekstraksi kata kunci. Pertama, mari instal pustaka ini jika belum ada: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Fungsionalitas utama tersedia dari objek `Rake`, yang bisa kita sesuaikan menggunakan beberapa parameter. Dalam kasus kami, kami akan menetapkan panjang minimum sebuah kata kunci menjadi 5 karakter, frekuensi minimum sebuah kata kunci dalam dokumen menjadi 3, dan jumlah maksimum kata dalam sebuah kata kunci - menjadi 2. Silakan coba nilai lain dan amati hasilnya.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Kami memperoleh daftar istilah beserta derajat kepentingan yang terkait. Seperti yang Anda lihat, disiplin ilmu paling relevan, seperti pembelajaran mesin dan big data, hadir dalam daftar pada posisi teratas.

## Langkah 4: Memvisualisasikan Hasil

Orang dapat menginterpretasikan data dengan lebih baik dalam bentuk visual. Oleh karena itu, sering kali masuk akal untuk memvisualisasikan data guna menarik beberapa wawasan. Kita dapat menggunakan pustaka `matplotlib` di Python untuk memplot distribusi sederhana dari kata kunci dengan relevansinya:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Namun, ada cara yang lebih baik untuk memvisualisasikan frekuensi kata - menggunakan **Word Cloud**. Kita perlu menginstal perpustakaan lain untuk memplot word cloud dari daftar kata kunci kita.


In [ ]:
!{sys.executable} -m pip install wordcloud

Objek `WordCloud` bertanggung jawab untuk menerima teks asli, atau daftar kata yang sudah dihitung frekuensinya, dan mengembalikan sebuah gambar, yang kemudian dapat ditampilkan menggunakan `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Kita juga bisa memasukkan teks asli ke `WordCloud` - mari kita lihat apakah kita bisa mendapatkan hasil yang serupa:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Anda dapat melihat bahwa word cloud sekarang terlihat lebih mengesankan, tetapi juga mengandung banyak gangguan (misalnya kata-kata yang tidak terkait seperti `Retrieved on`). Selain itu, kita mendapatkan lebih sedikit kata kunci yang terdiri dari dua kata, seperti *data scientist*, atau *computer science*. Ini karena algoritma RAKE melakukan pekerjaan yang jauh lebih baik dalam memilih kata kunci yang baik dari teks. Contoh ini menggambarkan pentingnya pra-pemrosesan dan pembersihan data, karena gambaran yang jelas pada akhirnya akan memungkinkan kita membuat keputusan yang lebih baik.

Dalam latihan ini kami telah melalui proses sederhana untuk mengekstrak beberapa makna dari teks Wikipedia, dalam bentuk kata kunci dan word cloud. Contoh ini cukup sederhana, tetapi menunjukkan dengan baik semua langkah khas yang akan diambil seorang data scientist saat bekerja dengan data, mulai dari akuisisi data hingga visualisasi.

Dalam kursus kami akan membahas semua langkah tersebut secara rinci. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
